<p style="font-size:16pt; font-weight:bold; color:red; padding-bottom:20px; float:right">Please rename this file before editing!</p>

# Session 12 — A Worked Example: Word Ladders

In 1877 Lewis Carroll, the author of *Alice in Wonderland*, invented a puzzle.
Change one word into another, one letter at a time, so that every step is a
word:

```text
COLD → CORD → CARD → WARD → WARM
```

He called it a word ladder. Is there always a ladder? What is the shortest
one? Which words can never be reached?

Nothing in the puzzle mentions a graph. That is the point of this notebook:
**the graph is something you decide to see.** Make every word a node, join two
words when they differ in exactly one letter, and "shortest ladder" becomes
"shortest path", which notebook 01 already solved.

| Step | What happens |
|---|---|
| 1 | The words |
| 2 | What is an edge? Building the graph, slowly and then quickly |
| 3 | Looking at the graph: degrees, hubs, loners |
| 4 | Ladders: breadth-first search |
| 5 | The parts of the graph |
| 6 | How far apart can two words be? |
| 7 | Checking against networkx |

> **Run this on ARC, with the kernel set to Conda Python 3.12.** The word list
> `four_letter_words.txt` is in this folder. The notebook takes about ten
> seconds, half of it in Step 6.

In [ ]:
import time
from collections import deque

import matplotlib.pyplot as plt
import networkx as nx

---

## Step 1 — The words

The four-letter words of a standard English word list: lower case, no names.

In [ ]:
with open("four_letter_words.txt", encoding="utf-8") as handle:
    words = handle.read().split()

print(len(words), "words")
print(words[:12], "...", words[-6:])

---

## Step 2 — What is an edge?

Two words are joined when they differ in exactly one position.

In [ ]:
def one_apart(first, second):
    """True when two words of the same length differ in exactly one letter."""
    differences = 0
    for a, b in zip(first, second):
        if a != b:
            differences += 1
    return differences == 1


print(one_apart("cold", "cord"), one_apart("cold", "card"), one_apart("cold", "cold"))

The obvious way to build the graph: compare every word with every other word.
Time it on the first 300 words before committing to all of them.

In [ ]:
start = time.perf_counter()
pairs = 0
for first in words[:300]:
    for second in words:
        if one_apart(first, second):
            pairs += 1
seconds = time.perf_counter() - start

print("300 words against all {:,}: {:.2f} s".format(len(words), seconds))
print("all of them would take about {:.0f} s, for {:,} comparisons".format(
    seconds / 300 * len(words), len(words) ** 2))

A few seconds, which is bearable for 2,442 words. But the work grows with the
*square* of the number of words: ten times the words, a hundred times the
wait. For the 100,000 words of the full list it would take hours. This is the
loop inside a loop of Session 11.

There is a better idea, and it is a hashing idea. Two words are one letter
apart exactly when they become **the same pattern** after one position is
blanked out:

```text
cold → _old  c_ld  co_d  col_
cord → _ord  c_rd  co_d  cor_
                    ↑ the same
```

So: put every word into four buckets, one per pattern, in a dictionary. Words
that share a bucket are neighbours. Nobody is compared with a word that could
not possibly match.

In [ ]:
start = time.perf_counter()

buckets = {}
for word in words:
    for position in range(len(word)):
        pattern = word[:position] + "_" + word[position + 1:]
        buckets.setdefault(pattern, []).append(word)

graph = {word: set() for word in words}          # every word is a node
for group in buckets.values():
    for first in group:
        for second in group:
            if first != second:
                graph[first].add(second)

print("built in {:.3f} s".format(time.perf_counter() - start))
print(len(buckets), "patterns; the fullest is",
      max(buckets, key=lambda pattern: len(buckets[pattern])),
      buckets[max(buckets, key=lambda pattern: len(buckets[pattern]))])

A hundredth of a second in place of several. The dictionary did the matching:
each word was filed four times and found its neighbours waiting there.

Note the line `graph = {word: set() for word in words}`. It makes every word a
node **before** any edge is added, so that a word with no neighbours at all is
still in the graph. Notebook 01 called forgetting this the most common bug in
a first graph program.

In [ ]:
edges = sum(len(neighbours) for neighbours in graph.values()) // 2
print("{:,} nodes, {:,} edges".format(len(graph), edges))
print("cold:", sorted(graph["cold"]))

Every edge is stored from both ends, so the sum of the degrees is twice the
number of edges.

**✏️ Try it:** What are the neighbours of "warm"? Of "ugly"? Guess before you look.

---

## Step 3 — Looking at the graph

Before searching, look. How many neighbours does a word have?

In [ ]:
degrees = {word: len(graph[word]) for word in graph}
values = sorted(degrees.values())

print("fewest", values[0], " median", values[len(values) // 2],
      " mean {:.1f}".format(sum(values) / len(values)), " most", values[-1])

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.hist(values, bins=range(0, max(values) + 2), color="steelblue", edgecolor="white")
ax.set_xlabel("neighbours of a word (its degree)")
ax.set_ylabel("words")
ax.set_title("Most words have a handful of neighbours; a few have two dozen")
fig.tight_layout()

In [ ]:
hubs = sorted(graph, key=lambda word: (-degrees[word], word))[:5]
for word in hubs:
    print("{} ({}): {}".format(word, degrees[word], " ".join(sorted(graph[word]))))

The best-connected words are made of common letters in common positions:
change any letter of "ware" and you are likely to land on another word.

And the other end:

In [ ]:
loners = sorted(word for word in graph if degrees[word] == 0)
print(len(loners), "words have no neighbour at all:")
print(" ".join(loners))

No ladder can start or end at any of these. Some are words with unusual
letters in unusual places: nothing is one letter away from "ugly". Others are
not really words: "abbr", "acct", "tbsp". The list has abbreviations in it,
and the graph has just pointed them out.

---

## Step 4 — Ladders

A shortest ladder is a shortest path. The function is the one from notebook
01, unchanged: it never knew it was about friends.

In [ ]:
def shortest_path(graph, start, target):
    """Return a path with the fewest steps, or None."""
    if start not in graph or target not in graph:
        return None
    came_from = {start: None}
    queue = deque([start])
    while queue:
        node = queue.popleft()
        if node == target:
            break
        for neighbour in sorted(graph[node]):
            if neighbour not in came_from:
                came_from[neighbour] = node
                queue.append(neighbour)
    if target not in came_from:
        return None
    path = []
    node = target
    while node is not None:
        path.append(node)
        node = came_from[node]
    path.reverse()
    return path


print(" → ".join(shortest_path(graph, "cold", "warm")))

Four steps, and Carroll's own answer. It is not the only one with four steps;
the search returns the first it finds, which depends on the alphabetical order
of the neighbours.

In [ ]:
for start, target in [("love", "hate"), ("lead", "gold"), ("head", "tail"),
                      ("fish", "bird"), ("slow", "fast"), ("work", "play"),
                      ("wolf", "lamb"), ("ugly", "cute")]:
    ladder = shortest_path(graph, start, target)
    if ladder is None:
        print("{} to {}: no ladder".format(start, target))
    else:
        print("{} to {}: {} steps   {}".format(start, target, len(ladder) - 1, " → ".join(ladder)))

Lead does turn into gold, in three steps. From "ugly" there is no ladder to
anywhere, as Step 3 predicted.

How far is everything from "cold"? The other search of notebook 01:

In [ ]:
def hop_distances(graph, start):
    hops = {start: 0}
    queue = deque([start])
    while queue:
        node = queue.popleft()
        for neighbour in graph[node]:
            if neighbour not in hops:
                hops[neighbour] = hops[node] + 1
                queue.append(neighbour)
    return hops


from_cold = hop_distances(graph, "cold")
print("{:,} of {:,} words can be reached from 'cold'".format(len(from_cold), len(graph)))

rings = {}
for steps in from_cold.values():
    rings[steps] = rings.get(steps, 0) + 1

fig, ax = plt.subplots(figsize=(7.5, 3.4))
ax.bar(sorted(rings), [rings[steps] for steps in sorted(rings)], color="steelblue")
ax.set_ylim(bottom=0)
ax.set_xlabel("steps from 'cold'")
ax.set_ylabel("words")
ax.set_title("Nearly every reachable word is within seven steps of 'cold'")
fig.tight_layout()

The rings grow quickly, peak at five steps, and thin out. Twelve words are one
step away and most of the vocabulary is four to six steps away. This shape,
where a few steps take you almost everywhere, is typical of real networks and
is what "six degrees of separation" is about.

**✏️ Try it:** Find the word farthest from "cold", and print the ladder to it.

---

## Step 5 — The parts of the graph

Not every word was reached from "cold". Which words hang together?

In [ ]:
def components(graph):
    seen = set()
    parts = []
    for node in graph:
        if node not in seen:
            part = set(hop_distances(graph, node))
            seen |= part
            parts.append(part)
    parts.sort(key=lambda part: (-len(part), min(part)))
    return parts


parts = components(graph)
print(len(parts), "parts, of sizes", [len(part) for part in parts[:10]], "...")
print("{:.0%} of the words are in the largest".format(len(parts[0]) / len(graph)))

One giant component with 94 % of the words, and a scatter of small ones: 81 of
them are the loners of Step 3. Look at the ones in between.

In [ ]:
for part in parts[1:7]:
    print(len(part), " ".join(sorted(part)))

The second largest is a surprise: **Roman numerals**. The word list contains
"xvii", "xxiv", "lxix" and their relatives, which are one letter apart from
each other and from nothing else. They form their own little world.

That is a finding about the *data*, and the graph delivered it unasked. Nobody
went looking for Roman numerals; a look at the components showed a cluster
that did not belong. Whether they should be in a list of words is a cleaning
question (Session 8), and now you know to ask it.

The next ones are tidy little islands: "ache, achy, acme, acne, acre, ashy"
can reach each other and nothing else.

---

## Step 6 — How far apart can two words be?

Within the giant component every word can reach every other. What is the
longest *shortest* ladder: the two words that are farthest apart?

There is no shortcut that is simple. Search from every word, and keep the
largest distance seen. That is 2,297 breadth-first searches; it takes a few
seconds.

In [ ]:
start = time.perf_counter()

giant = parts[0]
longest = 0
ends = None
for word in sorted(giant):
    hops = hop_distances(graph, word)
    farthest = max(hops, key=lambda other: (hops[other], other))
    if hops[farthest] > longest:
        longest = hops[farthest]
        ends = (word, farthest)

print("searched from {:,} words in {:.0f} s".format(len(giant), time.perf_counter() - start))
print("the two words farthest apart:", ends, "at", longest, "steps")
print(" → ".join(shortest_path(graph, ends[0], ends[1])))

Seventeen steps, and no shorter ladder exists between those two. The largest
distance in a graph is called its **diameter**.

Compare the two numbers of this notebook. The diameter is 17, and yet Step 4
showed that most words are within five or six steps of "cold". A few words sit
at the end of long thin chains, and they set the record; the typical distance
is far smaller. Session 9 again: the maximum describes the extreme, not the
usual.

That search-from-everywhere is the expensive kind of question. One search
costs about as much as reading the graph once. Searching from every node costs
the number of nodes times that: fine for 2,442 words, hopeless for a social
network of a billion people.

---

## Step 7 — Checking against networkx

The same graph, handed to the library, and the same questions.

In [ ]:
G = nx.Graph()
G.add_nodes_from(graph)
G.add_edges_from((word, neighbour) for word in graph for neighbour in graph[word])

print(G.number_of_nodes(), "nodes,", G.number_of_edges(), "edges")
assert nx.shortest_path_length(G, "cold", "warm") == len(shortest_path(graph, "cold", "warm")) - 1
assert dict(nx.single_source_shortest_path_length(G, "cold")) == from_cold
assert sorted(len(part) for part in nx.connected_components(G)) == sorted(len(part) for part in parts)
assert not nx.has_path(G, "ugly", "cute")
print("networkx agrees on the ladder, the distances and the components")

In [ ]:
ladders = list(nx.all_shortest_paths(G, "cold", "warm"))
print(len(ladders), "different ladders of four steps from cold to warm:")
for ladder in ladders:
    print("  ", " → ".join(ladder))

The library also answers a question the hand-written search could not: *all*
the shortest ladders, not just one. Ours returned the first of these.

### What this example was about

| Step | The move |
|---|---|
| 2 | Decide what a node is and what an edge means. The puzzle did not say "graph". |
| 2 | Build the edges with a dictionary of buckets, not by comparing all pairs. |
| 3 | Look at degrees before searching: hubs and loners explain later results. |
| 4 | Shortest ladder = shortest path. The search is the same for friends, airports and words. |
| 5 | Look at the components: they show what cannot be reached, and odd things in the data. |
| 6 | "Farthest apart" means searching from everywhere, and costs accordingly. |
| 7 | Check your own search against the library, then use the library. |

---

## Exercises

### 🧩 Exercise 1 — Your own ladders

Find the shortest ladders for five pairs of words of your choice. Find a pair
that needs at least eight steps, and a pair with no ladder where neither word
is a loner. (For the second: pick two words from different components.)

### 🧩 Exercise 2 — Without the numerals

Remove the Roman numerals from `words` (they contain only the letters i, v, x,
l, c, d and m; decide what to do about real words such as "mild" and "dill"
that do too), rebuild the graph, and compare the number of components before
and after.

### 🧩 Exercise 3 — A directed version

Make the ladder one-way: a step is allowed only if it moves a letter *later*
in the alphabet ("cold" to "cord", not back). Build that directed graph. From
"aces", how many words can be reached? Can you get from "cold" to "warm"?
Remember that every word must still be a key.

### 🧩 Exercise 4 — Weighted ladders

Give each step a cost: 1 if the changed letter is a vowel on both sides, 2
otherwise. Build `G` in networkx with those weights and compare
`nx.shortest_path(G, "work", "play")` with and without `weight=`. Do the two
ladders differ?

### 🧩 Exercise 5 — Five letters

The system list `/usr/share/dict/words` has about 100,000 words. Extract the
five-letter ones the way `four_letter_words.txt` was made (lower case, the
letters a to z only), build the graph with the bucket method, and report the
number of nodes, edges and components, and the size of the largest. Is the
five-letter graph better or worse connected than the four-letter one?

---

**Next:** `03_Graphs_Reference` lists the tools used here, by task. Homework
11 runs the same searches on airline routes, where edges have a direction and
a length.